# ForgeSight AI — YOLOv8n Fine-Tuning Notebook

Fine-tunes a YOLOv8n base model on the ForgeSight synthetic dataset (or, later, a real labeled PCB defect dataset that follows the same directory/`data.yaml` layout) and saves the best checkpoint to the exact path `config/vision/vision.yaml` expects.

**Before running this notebook:**
```bash
python scripts/generate_synthetic_vision_dataset.py --num-images 200 --val-split 0.2
```
This must be run from the repo root so `data/vision/synthetic_dataset/data.yaml` exists.

**To later fine-tune on a real dataset instead:** point `DATASET_YAML_PATH` at your real dataset's `data.yaml` (same YOLO format: `train`/`val` image dirs + matching label dirs + `nc`/`names`), update `OUTPUT_CHECKPOINT_NAME` and `DATASET_DISCLOSURE_STRING` below, and re-run from the Configuration cell onward. No other code needs to change.

## 1. Setup

In [ ]:
!pip install -q ultralytics>=8.2.0 opencv-python-headless>=4.9.0 pillow>=10.3.0 numpy>=1.26.0

In [ ]:
import os
import shutil
from pathlib import Path
from datetime import datetime, timezone

import torch
import yaml
from ultralytics import YOLO

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

## 2. Configuration

Everything you're likely to change lives in this one cell. When you later fine-tune on a real dataset, this is the only cell you should need to edit.

In [ ]:
# --- Path resolution: assumes this notebook lives in <repo_root>/notebooks/ ---
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

# --- Dataset config ---
# Synthetic dataset generated by scripts/generate_synthetic_vision_dataset.py.
# To fine-tune on a REAL dataset later: change this path to your real dataset's
# data.yaml (same YOLO layout: images/{train,val}, labels/{train,val}, nc, names).
DATASET_YAML_PATH = REPO_ROOT / "data" / "vision" / "synthetic_dataset" / "data.yaml"

# A short, honest string describing what this checkpoint was trained on.
# This value MUST be copied into config/vision/vision.yaml's
# model.dataset_used_for_training field after training, so every CvFinding
# produced by this checkpoint carries an accurate disclosure.
DATASET_DISCLOSURE_STRING = "synthetic-placeholder-v1 (not for production use)"

# --- Base model ---
# yolov8n.pt = Ultralytics' smallest pretrained COCO checkpoint, used only as
# a starting point for transfer learning. Its original COCO classes are
# irrelevant here; fine-tuning replaces the detection head for our N classes.
BASE_MODEL = "yolov8n.pt"

# --- Training hyperparameters ---
EPOCHS = 5              # small on purpose for the synthetic fixture dataset;
                         # raise substantially (50-300) for a real dataset
IMAGE_SIZE = 640         # must match config/vision/vision.yaml -> model.image_size
BATCH_SIZE = 16
PATIENCE = 20            # early-stopping patience (epochs with no val improvement)

# --- Output ---
RUN_NAME = "forgesight_yolov8n_finetune"
RUNS_DIR = REPO_ROOT / "runs" / "vision"
OUTPUT_CHECKPOINT_DIR = REPO_ROOT / "models" / "vision" / "checkpoints"
OUTPUT_CHECKPOINT_NAME = "yolov8-forgesight-synthetic-v1.pt"  # must match config/vision/vision.yaml

print("Repo root:            ", REPO_ROOT)
print("Dataset YAML:         ", DATASET_YAML_PATH)
print("Output checkpoint at: ", OUTPUT_CHECKPOINT_DIR / OUTPUT_CHECKPOINT_NAME)

assert DATASET_YAML_PATH.exists(), (
    f"Dataset YAML not found at {DATASET_YAML_PATH}. "
    "Run scripts/generate_synthetic_vision_dataset.py first, or point "
    "DATASET_YAML_PATH at your own dataset."
)

## 3. Inspect the dataset config

In [ ]:
with open(DATASET_YAML_PATH, "r") as f:
    dataset_config = yaml.safe_load(f)

print("Number of classes:", dataset_config["nc"])
print("Class names:", dataset_config["names"])

train_images_dir = Path(dataset_config["path"]) / dataset_config["train"]
val_images_dir = Path(dataset_config["path"]) / dataset_config["val"]
print(f"Train images: {len(list(train_images_dir.glob('*.jpg')))}")
print(f"Val images:   {len(list(val_images_dir.glob('*.jpg')))}")

## 4. Load base model and fine-tune

In [ ]:
model = YOLO(BASE_MODEL)

training_started_at = datetime.now(timezone.utc)
print(f"Training started at {training_started_at.isoformat()}")

results = model.train(
    data=str(DATASET_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
)

print(f"Training finished. Duration: {datetime.now(timezone.utc) - training_started_at}")

## 5. Validate the best checkpoint

In [ ]:
best_checkpoint_path = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
assert best_checkpoint_path.exists(), f"Expected best checkpoint at {best_checkpoint_path}, not found."

validation_model = YOLO(str(best_checkpoint_path))
metrics = validation_model.val(data=str(DATASET_YAML_PATH), imgsz=IMAGE_SIZE, device=DEVICE)

print("mAP50:    ", metrics.box.map50)
print("mAP50-95: ", metrics.box.map)
print(
    "\nNote: on the synthetic fixture dataset these numbers only confirm the "
    "pipeline learns to find its own visually-distinct marker box — they say "
    "nothing about real-world PCB defect detection accuracy."
)

## 6. Sanity-check inference on a sample image

In [ ]:
sample_image_path = next(val_images_dir.glob("*.jpg"))
prediction = validation_model.predict(str(sample_image_path), imgsz=IMAGE_SIZE, device=DEVICE, conf=0.25)

for box in prediction[0].boxes:
    class_idx = int(box.cls.item())
    confidence = float(box.conf.item())
    xyxy = box.xyxy[0].tolist()
    class_name = dataset_config["names"][class_idx]
    print(f"detected: {class_name}  confidence={confidence:.3f}  box={xyxy}")

prediction[0].save(filename=str(REPO_ROOT / "notebooks" / "sample_prediction.jpg"))
print("Annotated sample saved to notebooks/sample_prediction.jpg")

## 7. Copy the best checkpoint into the ForgeSight checkpoints directory

This writes the file exactly where `config/vision/vision.yaml` (`model.checkpoint_path`) expects it.

In [ ]:
OUTPUT_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
destination_path = OUTPUT_CHECKPOINT_DIR / OUTPUT_CHECKPOINT_NAME

shutil.copy2(best_checkpoint_path, destination_path)
print(f"Copied {best_checkpoint_path} -> {destination_path}")

checkpoint_size_mb = destination_path.stat().st_size / (1024 * 1024)
print(f"Checkpoint size: {checkpoint_size_mb:.1f} MB")

## 8. Final checklist before using this checkpoint in ForgeSight

Run this cell to confirm `config/vision/vision.yaml` is consistent with what you just trained. If any assertion fails, update the YAML file (or the variables in Section 2) and re-run.

In [ ]:
vision_yaml_path = REPO_ROOT / "config" / "vision" / "vision.yaml"
with open(vision_yaml_path, "r") as f:
    vision_config = yaml.safe_load(f)

checks = {
    "checkpoint_path matches destination": (
        Path(vision_config["model"]["checkpoint_path"]).name == OUTPUT_CHECKPOINT_NAME
    ),
    "dataset_used_for_training is set and non-empty": bool(
        vision_config["model"]["dataset_used_for_training"].strip()
    ),
    "defect_classes count matches trained nc": (
        len(vision_config["defect_classes"]) == dataset_config["nc"]
    ),
    "defect_classes order matches trained class order": (
        vision_config["defect_classes"] == dataset_config["names"]
    ),
    "image_size matches training imgsz": (
        vision_config["model"]["image_size"] == IMAGE_SIZE
    ),
}

for check_name, passed in checks.items():
    status = "PASS" if passed else "FAIL - fix config/vision/vision.yaml before deploying this checkpoint"
    print(f"[{status}] {check_name}")

print(
    "\nIMPORTANT: if this checkpoint was trained on the synthetic fixture "
    "dataset, leave dataset_used_for_training exactly as \n"
    f"  '{DATASET_DISCLOSURE_STRING}'\n"
    "When you later fine-tune on a REAL labeled defect dataset, update "
    "dataset_used_for_training in config/vision/vision.yaml to accurately "
    "describe that real dataset (name/version/source) BEFORE deploying the "
    "new checkpoint, so every CvFinding it produces carries an honest "
    "provenance record."
)